# Checkpoint 1 — UCI SECOM acquisition and data-quality audit

**Purpose:** establish provenance and understand the raw data before choosing a split, preprocessing rule, feature set, or model. This notebook intentionally stops before modeling.

Official source: [UCI SECOM dataset](https://archive.ics.uci.edu/dataset/179/secom) · Dataset DOI: [10.24432/C54305](https://doi.org/10.24432/C54305)

> Scope boundary: SECOM is an anonymized public dataset donated in 2008. It is useful for demonstrating analytical discipline, not for making claims about current fabs, specific equipment, physical root causes, or production readiness.

## What you should learn at this checkpoint

By the end, you should be able to explain:

1. why source URL and file hash are part of reproducibility;
2. how the source encodes pass/fail and where timestamps live;
3. why 104 failures out of 1,567 examples is an operating-threshold problem, not an accuracy contest;
4. why missingness and constant columns must be handled inside the training workflow later;
5. why auditing all rows is acceptable for description, but using full-data summaries to choose features would leak validation information.

The repository pins a tested Python environment in `requirements.txt`. Colab already supplies the standard libraries used here, so this notebook prints versions rather than modifying the managed runtime.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZIP_DEFLATED, ZipFile
import json
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:  # Allows the same code to run as a plain Python script.
    display = print

pd.set_option("display.max_colwidth", 120)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## 1. Acquire the official archive and verify provenance

We download from UCI's static archive—not from Kaggle or an unknown mirror. The expected SHA-256 digest pins the exact bytes used for this case study. If UCI changes the archive, the cell fails loudly so we can inspect and document the change rather than silently analyzing different data. No secret or credential is required.

In [ ]:
UCI_DATASET_PAGE = "https://archive.ics.uci.edu/dataset/179/secom"
UCI_ARCHIVE_URL = "https://archive.ics.uci.edu/static/public/179/secom.zip"
EXPECTED_ARCHIVE_SHA256 = "eea568baf3c2229096d7d294cf0b096b5502bd96d92c0b80a65b84714059be8e"
REQUIRED_MEMBERS = {"secom.data", "secom_labels.data", "secom.names"}

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

archive_path = RAW_DIR / "secom.zip"

def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not archive_path.exists() or file_sha256(archive_path) != EXPECTED_ARCHIVE_SHA256:
    print("Downloading the official UCI archive...")
    urlretrieve(UCI_ARCHIVE_URL, archive_path)

observed_archive_sha256 = file_sha256(archive_path)
if observed_archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "The official archive bytes do not match the expected SHA-256. "
        "Stop and investigate the source before continuing."
    )

with ZipFile(archive_path) as archive:
    archive_members = set(archive.namelist())
    missing_members = REQUIRED_MEMBERS - archive_members
    if missing_members:
        raise RuntimeError(f"Official archive is missing: {sorted(missing_members)}")
    for member in sorted(REQUIRED_MEMBERS):
        # Extract only the three exact expected filenames.
        destination = RAW_DIR / member
        destination.write_bytes(archive.read(member))

print(f"Verified archive: {archive_path}")
print(f"SHA-256: {observed_archive_sha256}")
print(f"Members: {sorted(REQUIRED_MEMBERS)}")

## 2. Parse measurements, labels, and timestamps

The raw measurement file is whitespace-delimited, has no header, and represents missing values as `NaN`. We assign neutral names (`feature_000`, …) because the physical meanings are not provided.

The label file contains the source label and a quoted test-point timestamp. UCI defines `-1` as pass and `1` as fail. We preserve the source label and create a clearly named `is_fail` target where fail becomes `1` and pass becomes `0`.

In [ ]:
X = pd.read_csv(
    RAW_DIR / "secom.data",
    sep=r"\s+",
    header=None,
    na_values=["NaN"],
    dtype=float,
)
X.columns = [f"feature_{index:03d}" for index in range(X.shape[1])]

labels = pd.read_csv(
    RAW_DIR / "secom_labels.data",
    sep=r"\s+",
    header=None,
    names=["source_label", "timestamp_text"],
    quotechar='"',
)
labels["source_label"] = labels["source_label"].astype(int)
labels["timestamp"] = pd.to_datetime(
    labels["timestamp_text"],
    format="%d/%m/%Y %H:%M:%S",
    errors="raise",
)
labels["is_fail"] = (labels["source_label"] == 1).astype(int)

print(f"Measurement matrix: {X.shape}")
print(f"Label/timestamp table: {labels.shape}")
display(labels.head())

## 3. Verify structural expectations

The audit distinguishes a failed check from a documented source discrepancy. UCI's catalog and `secom.names` say **591 features**, but the current raw `secom.data` file contains **590 numeric columns**. We do not invent a column to force agreement. The separate timestamp could explain the count, but without a definitive data dictionary that remains only a hypothesis.

In [ ]:
source_label_counts = labels["source_label"].value_counts().sort_index()

checks = pd.DataFrame(
    [
        {"check": "Measurement rows", "observed": len(X), "expected": 1567, "status": "PASS" if len(X) == 1567 else "FAIL"},
        {"check": "Label rows align", "observed": len(labels), "expected": len(X), "status": "PASS" if len(labels) == len(X) else "FAIL"},
        {"check": "Raw numeric columns", "observed": X.shape[1], "expected": "UCI catalog: 591", "status": "NOTE: raw file has 590" if X.shape[1] == 590 else "REVIEW"},
        {"check": "Source label values", "observed": sorted(labels["source_label"].unique().tolist()), "expected": [-1, 1], "status": "PASS" if set(labels["source_label"]) == {-1, 1} else "FAIL"},
        {"check": "Failure count", "observed": int(source_label_counts.get(1, 0)), "expected": 104, "status": "PASS" if int(source_label_counts.get(1, 0)) == 104 else "FAIL"},
        {"check": "Timestamp parse failures", "observed": int(labels["timestamp"].isna().sum()), "expected": 0, "status": "PASS" if labels["timestamp"].notna().all() else "FAIL"},
        {"check": "Timestamps nondecreasing", "observed": bool(labels["timestamp"].is_monotonic_increasing), "expected": True, "status": "PASS" if labels["timestamp"].is_monotonic_increasing else "FAIL"},
    ]
)

display(checks)

if (checks["status"] == "FAIL").any():
    raise AssertionError("A structural data check failed; do not continue to modeling.")

In [ ]:
label_summary = pd.DataFrame(
    {
        "source_label": [-1, 1],
        "meaning": ["pass", "fail"],
        "count": [int(source_label_counts[-1]), int(source_label_counts[1])],
    }
)
label_summary["share"] = label_summary["count"] / len(labels)

timestamp_summary = pd.Series(
    {
        "minimum": labels["timestamp"].min(),
        "maximum": labels["timestamp"].max(),
        "nondecreasing": labels["timestamp"].is_monotonic_increasing,
        "repeated timestamps after first occurrence": int(labels["timestamp"].duplicated().sum()),
        "repeated (timestamp, label) pairs after first occurrence": int(labels.duplicated(["timestamp", "source_label"]).sum()),
    },
    name="value",
)

label_summary_display = label_summary.copy()
label_summary_display["share"] = label_summary_display["share"].map(lambda value: f"{value:.2%}")
display(label_summary_display)
display(timestamp_summary.to_frame())

### Interpretation checkpoint

An always-pass classifier would be correct on 1,463 of 1,567 records—**93.36% accuracy**—while detecting zero failures. That is why later evaluation will center precision-recall behavior, failure recall, false alarms/review load, and raw confusion-matrix counts rather than plain accuracy. No model has been fit yet.

## 4. Audit missingness, constants, and duplicates

These are descriptive full-dataset summaries. They help us understand the problem, but the resulting table will **not** be used to remove columns before validation. Any later missingness threshold, imputer, constant filter, scaler, or supervised feature selector must be learned from training data only.

In [ ]:
missing_mask = X.isna()
non_null_unique = X.nunique(dropna=True)

feature_quality = pd.DataFrame(
    {
        "feature": X.columns,
        "missing_count": missing_mask.sum(axis=0).to_numpy(),
        "missing_rate": missing_mask.mean(axis=0).to_numpy(),
        "non_null_unique": non_null_unique.to_numpy(),
    }
)
feature_quality["is_constant_or_empty"] = feature_quality["non_null_unique"] <= 1
feature_quality = feature_quality.sort_values(
    ["missing_rate", "feature"], ascending=[False, True]
).reset_index(drop=True)

row_missing_rate = missing_mask.mean(axis=1)
total_missing = int(missing_mask.to_numpy().sum())
total_cells = int(X.size)

quality_metrics = {
    "rows": int(X.shape[0]),
    "raw_numeric_features": int(X.shape[1]),
    "official_catalog_feature_count": 591,
    "passes": int((labels["source_label"] == -1).sum()),
    "failures": int((labels["source_label"] == 1).sum()),
    "failure_rate": float(labels["is_fail"].mean()),
    "total_cells": total_cells,
    "missing_cells": total_missing,
    "overall_missing_rate": total_missing / total_cells,
    "features_with_any_missing": int((feature_quality["missing_rate"] > 0).sum()),
    "features_over_20pct_missing": int((feature_quality["missing_rate"] > 0.20).sum()),
    "features_over_50pct_missing": int((feature_quality["missing_rate"] > 0.50).sum()),
    "constant_or_empty_features": int(feature_quality["is_constant_or_empty"].sum()),
    "exact_duplicate_measurement_rows": int(X.duplicated().sum()),
    "median_row_missing_rate": float(row_missing_rate.median()),
    "maximum_row_missing_rate": float(row_missing_rate.max()),
    "repeated_timestamps_after_first": int(labels["timestamp"].duplicated().sum()),
}

quality_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in quality_metrics.items()]
)
display(quality_table)
top_quality_display = feature_quality.head(15).copy()
top_quality_display["missing_rate"] = top_quality_display["missing_rate"].map(lambda value: f"{value:.2%}")
display(top_quality_display)

## 5. Two compact diagnostic figures

The first makes the rare-event problem visible. The second shows the 20 anonymous variables with the highest missingness; it is a data-quality view, **not** a feature-importance chart.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
bars = ax.bar(
    label_summary["meaning"].str.title(),
    label_summary["count"],
    color=["#4C78A8", "#E45756"],
    width=0.62,
)
for bar, count, share in zip(bars, label_summary["count"], label_summary["share"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 25,
        f"{count:,}  ({share:.2%})",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax.set_title("SECOM outcome imbalance", loc="left", fontweight="bold")
ax.set_ylabel("Production records")
ax.set_ylim(0, 1600)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
class_balance_path = FIGURE_DIR / "class_balance.png"
fig.savefig(class_balance_path, dpi=180, bbox_inches="tight")
plt.show()

top_missing = feature_quality.head(20).sort_values("missing_rate")
fig, ax = plt.subplots(figsize=(8.4, 6.6))
ax.barh(
    top_missing["feature"],
    top_missing["missing_rate"] * 100,
    color="#F2A541",
)
ax.set_title("Highest feature missingness rates", loc="left", fontweight="bold")
ax.set_xlabel("Missing values (% of records)")
ax.set_ylabel("Anonymous feature")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
missingness_path = FIGURE_DIR / "top_feature_missingness.png"
fig.savefig(missingness_path, dpi=180, bbox_inches="tight")
plt.show()

## 6. Write the Checkpoint 1 data-quality report

The report records source hashes, package versions, exact counts, and cautions for the next checkpoint. It also creates a ZIP that is easy to download from Colab's Files sidebar.

In [ ]:
generated_at_utc = datetime.now(timezone.utc).isoformat()

quality_table_path = REPORT_DIR / "data_quality_summary.csv"
feature_quality_path = REPORT_DIR / "feature_quality.csv"
report_path = REPORT_DIR / "data_quality_report.md"
manifest_path = REPORT_DIR / "checkpoint_1_manifest.json"

quality_table.to_csv(quality_table_path, index=False)
feature_quality.to_csv(feature_quality_path, index=False)

raw_hashes = {
    filename: file_sha256(RAW_DIR / filename)
    for filename in sorted(REQUIRED_MEMBERS)
}
manifest = {
    "generated_at_utc": generated_at_utc,
    "dataset_page": UCI_DATASET_PAGE,
    "archive_url": UCI_ARCHIVE_URL,
    "archive_sha256": observed_archive_sha256,
    "raw_file_sha256": raw_hashes,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "quality_metrics": quality_metrics,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report_markdown = f"""# Checkpoint 1 data-quality report

Generated: {generated_at_utc}

## Provenance

- Official dataset page: {UCI_DATASET_PAGE}
- Official archive: {UCI_ARCHIVE_URL}
- Archive SHA-256: `{observed_archive_sha256}`
- Source labels: `-1 = pass`, `1 = fail`

## Structural findings

- Measurement matrix: **{X.shape[0]:,} rows × {X.shape[1]:,} raw numeric columns**.
- Source metadata lists 591 features, while the raw numeric file parses to 590 columns. This discrepancy is preserved and documented.
- Outcome: **{quality_metrics['passes']:,} passes** and **{quality_metrics['failures']:,} failures** ({quality_metrics['failure_rate']:.2%} fail).
- Timestamps: **{labels['timestamp'].min()}** through **{labels['timestamp'].max()}**, in nondecreasing file order.
- Repeated timestamps after first occurrence: **{quality_metrics['repeated_timestamps_after_first']:,}**. A timestamp collision alone does not prove duplicate production entities.

## Missingness and low-information columns

- Missing cells: **{quality_metrics['missing_cells']:,} / {quality_metrics['total_cells']:,} ({quality_metrics['overall_missing_rate']:.2%})**.
- Features with any missing value: **{quality_metrics['features_with_any_missing']:,}**.
- Features over 20% missing: **{quality_metrics['features_over_20pct_missing']:,}**.
- Features over 50% missing: **{quality_metrics['features_over_50pct_missing']:,}**.
- Constant or empty features when inspected descriptively: **{quality_metrics['constant_or_empty_features']:,}**.
- Exact duplicate measurement rows: **{quality_metrics['exact_duplicate_measurement_rows']:,}**.
- Median row missingness: **{quality_metrics['median_row_missing_rate']:.2%}**; maximum: **{quality_metrics['maximum_row_missing_rate']:.2%}**.

## Implications for the next checkpoint

1. A plain accuracy result can hide complete failure blindness because the pass class is 93.36% of the data.
2. Chronological validation is possible because timestamps parse and are nondecreasing, but the failure count in each proposed split must be checked before locking it.
3. Missingness handling, constant filtering, scaling, feature selection, calibration, and threshold tuning must be fit using training data only.
4. Full-data feature-quality rankings above are descriptive and must not become a pre-split feature-selection list.
5. Anonymous-variable coefficients or importances will be described as predictive associations, never physical root causes.

No model was fit in this checkpoint.
"""
report_path.write_text(report_markdown, encoding="utf-8")

output_zip = PROJECT_ROOT / "checkpoint_1_outputs.zip"
output_files = [
    quality_table_path,
    feature_quality_path,
    report_path,
    manifest_path,
    class_balance_path,
    missingness_path,
]
with ZipFile(output_zip, "w", compression=ZIP_DEFLATED) as bundle:
    for path in output_files:
        bundle.write(path, arcname=path.relative_to(PROJECT_ROOT))

print("Wrote:")
for path in output_files + [output_zip]:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")

## 7. Final checkpoint gate

The cell below is deliberately strict. If it prints `CHECKPOINT 1 COMPLETE`, stop here and return the requested values in this task. Do not begin modeling or remove columns yet.

In [ ]:
assert X.shape == (1567, 590)
assert source_label_counts.to_dict() == {-1: 1463, 1: 104}
assert total_missing == 41951
assert labels["timestamp"].is_monotonic_increasing
assert quality_metrics["constant_or_empty_features"] == 116
assert output_zip.exists()

print("=" * 72)
print("CHECKPOINT 1 COMPLETE — STOP BEFORE MODELING")
print("=" * 72)
print(f"1) Measurement matrix: {X.shape}")
print(f"2) Labels: {source_label_counts.to_dict()}  |  failure rate: {labels['is_fail'].mean():.2%}")
print(f"3) Timestamp range: {labels['timestamp'].min()} to {labels['timestamp'].max()}")
print(f"4) Missing cells: {total_missing:,} / {total_cells:,} ({total_missing / total_cells:.2%})")
print(f"5) Constant-or-empty features (descriptive): {quality_metrics['constant_or_empty_features']}")
print("6) UCI catalog/raw-file feature-count note: 591 listed vs 590 parsed")
print("\nReply in the project task with these six lines and any warning/error text.")

### Teach-back prompts (do not overthink them)

Before Checkpoint 2, try to answer in your own words:

- Why would 93.36% accuracy be a useless result here?
- Why do we keep the raw `-1/1` label and also create `is_fail`?
- Why can we describe full-dataset missingness now but cannot use this table to choose features before the split?
- What does the 591-versus-590 discrepancy teach us about production data pipelines?
- Why can an anonymous high-importance feature support triage but not a root-cause claim?

We will use your run confirmation—not memorized textbook answers—to choose the chronological split in Checkpoint 2.